The Dataset

In [1]:
!pip install pandas
!pip install numpy
!pip install scikit-learn

# Preprocessing

In [1]:
# Load data. 45 columns, 2789 rows
import numpy as np
import pandas as pd
rice =  pd.read_csv("data/paddydataset.csv")
print(rice.head(5))

   Hectares      Agriblock      Variety Soil Types  Seedrate(in Kg)  \
0          6     Cuddalore        CO_43   alluvial              150   
1          6   Kurinjipadi      ponmani       clay              150   
2          6       Panruti  delux ponni   alluvial              150   
3          6  Kallakurichi        CO_43       clay              150   
4          6  Sankarapuram      ponmani   alluvial              150   

   LP_Mainfield(in Tonnes) Nursery  Nursery area (Cents)  \
0                     75.0     dry                   120   
1                     75.0     wet                   120   
2                     75.0     dry                   120   
3                     75.0     wet                   120   
4                     75.0     dry                   120   

   LP_nurseryarea(in Tonnes)  DAP_20days  ...  Wind Direction_D1_D30  \
0                          6         240  ...                     SW   
1                          6         240  ...                     NW

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Find categorical columns so they can be One Hot Encoded
X_categoricals = rice.select_dtypes(include=['object']).columns.tolist()

# One Hot Encode categoricals so they can be used with quantitatives
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(rice[X_categoricals])
one_hot_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(X_categoricals))

rice_encoded = pd.concat([rice, one_hot_df], axis=1)
rice_encoded = rice_encoded.drop(X_categoricals, axis=1)

In [3]:
# Separate data into predictors and predictions

# This copy simplifies X and y separation
copy_r_encoded = rice_encoded
y = copy_r_encoded['Paddy yield(in Kg)']
X = copy_r_encoded.drop(['Paddy yield(in Kg)', 'Trash(in bundles)'], axis = 1)


In [4]:
# Split into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)
# Split training set into training and validation sets
#X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.2)

# Training

Either run the cell below to train the model, or load the already trained model.
Note that the training will take a long time.

In [14]:
# Reduce overfitting and # used features using Forward Feature Selection.
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
# The set of hyperparameters the RandomizedSearchCV will try
parameters = {
    'loss': ["squared_error", "poisson"],
    'activation': ['identity', 'logistic', 'tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'learning_rate': ['constant', 'adaptive'],
}

# This uses Multilayer Perceptron Regressor model.
regr = MLPRegressor(random_state=1, max_iter=1000, tol=0.1)


# Use forward feature selection to select 5 of the most useful features from the set
sfs = SequentialFeatureSelector(regr, n_features_to_select=5, direction='forward', n_jobs = -1)
sfs.fit(X_train, y_train)
# Reduce the dataset to only contain these features
X_train_selected = sfs.transform(X_train)
X_test_selected = sfs.transform(X_test)
# Use RandomizedSearchCV to find the optimal hyperparameters for the model, while being faster than GridSearchCV
sfs_cv = GridSearchCV(regr, parameters, cv = 5, n_jobs = -1)
sfs_cv.fit(X_train_selected, y_train)

C:\Users\justi\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
40 fits failed out of a total of 160.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\justi\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\justi\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\justi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py", line 849, in fit

,estimator,"MLPRegressor(...te=1, tol=0.1)"
,param_grid,"{'activation': ['identity', 'logistic', ...], 'learning_rate': ['constant', 'adaptive'], 'loss': ['squared_error', 'poisson'], 'solver': ['sgd', 'adam']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [24]:
print("Selected features:", sfs.get_feature_names_out())
print("Best params:", sfs_cv.best_params_)
print("Best score:", sfs_cv.best_score_)

Selected features: ['DAP_20days' 'Weed28D_thiobencarb' '30DAI(in mm)' '51_70DRain(in mm)'
 'Variety_delux ponni']
Best params: {'activation': 'relu', 'learning_rate': 'constant', 'loss': 'squared_error', 'solver': 'adam'}
Best score: 0.9917020849277642


Load Model

In [ ]:
# Load the model
import pickle
with open('model.pkl', 'rb') as f:
    sfs_cv = pickle.load(f)

# Save Model

In [16]:
# Save model because it takes a long time to train
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(sfs_cv, f)

# Scoring

In [25]:
# Use R Squared (coefficient of determination) to determine score. The MLPR's built in scorer is R squared.

y_score_train = sfs_cv.score(X_train_selected, y_train)
y_score_test = sfs_cv.score(X_test_selected, y_test)

print("Training R^2:", y_score_train)
print("Test R^2:", y_score_test)



Training R^2: 0.9914834049342887
Test R^2: 0.9915887522653513
